# SheafPatternFusion Phase 2.5 - WP2.5.1 degeneracy null battery

Scores nulls N0-N4 against the certificate labels on all engine-undecided rows of the frozen merge and emits the priority sample S* for the audit.

Code: pip-installed from hugogobato/sheafpatternfusion @ v0.3.0.
Runtime: CPU-only (~2 cores). Expected wall time: minutes.

The first cell installs the pinned package and restarts the kernel once (required after upgrading numpy/scipy in place). After the runtime reconnects, run Runtime > Run all again; the install cell detects the pins and skips.

In [ ]:
import importlib.metadata as md
import os
import subprocess
import sys

WANT = {'numpy': '2.4.3', 'scipy': '1.17.1', 'sheafpatternfusion': '0.3.0'}
TAG = 'v0.3.0'
REPO = 'https://github.com/hugogobato/sheafpatternfusion.git'


def _ver(pkg):
    try:
        return md.version(pkg)
    except Exception:
        return None


if all(_ver(p) == v for p, v in WANT.items()):
    print('environment OK:', WANT)
else:
    print('installing sheafpatternfusion@' + TAG + ' (one-time per session) ...')
    res = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                          f'git+{REPO}@{TAG}'])
    if res.returncode != 0:
        raise RuntimeError('pip install failed; see log above')
    print('installed -> restarting the kernel so the ABI-matched numpy/scipy')
    print('binaries load cleanly.')
    print('When the runtime reconnects, run Runtime > Run all again; this')
    print('cell will detect the pins and skip.')
    os.kill(os.getpid(), 9)


In [ ]:
import functools
import json
import multiprocessing as mp
import os
import pathlib
import time

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'


In [ ]:
import urllib.request
FROZEN_URL = 'https://raw.githubusercontent.com/hugogobato/sheafpatternfusion/v0.3.0/data/frozen/instances_merged.jsonl'
FROZEN_PATH = pathlib.Path('/content/instances_merged.jsonl')
if not FROZEN_PATH.exists():
    print('fetching frozen merge from', FROZEN_URL)
    urllib.request.urlretrieve(FROZEN_URL, FROZEN_PATH)
ROWS = [json.loads(l) for l in open(FROZEN_PATH)]
print('frozen merge:', len(ROWS), 'rows')


In [ ]:
BATTERY_CFG = json.loads(r'''{"description": "WP2.5.1 degeneracy null battery configuration. Frozen before any Phase-2.5 run.", "population": "engine-UNDETERMINED rows of data/frozen/instances_merged.jsonl", "tau_frac_observed": [0.3, 0.32, 0.34, 0.36, 0.38, 0.4, 0.42, 0.44, 0.46, 0.48, 0.5, 0.52, 0.54, 0.56, 0.58, 0.6, 0.62, 0.64, 0.66, 0.68, 0.7, 0.72, 0.74, 0.76, 0.78, 0.8, 0.82, 0.84, 0.86, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 1.0], "tau_overlap": [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0], "tau_width": [0.001, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45], "priority_sample_cap": 200, "outputs": ["results/phase25/null_battery.json", "results/phase25/null_battery.csv", "results/phase25/priority_sample.jsonl"]}''')


In [ ]:
UNDECIDED = [r for r in ROWS if r['gt_recoverable'].startswith('UNDETERMINED')]
print('undecided rows:', len(UNDECIDED))


In [ ]:

OUT_DIR = pathlib.Path('/content/results/phase25')
OUT_DIR.mkdir(parents=True, exist_ok=True)


def load_done(path, key_fn):
    done = set()
    if path.exists():
        for line in path.read_text().splitlines():
            try:
                rec = json.loads(line)
                done.add(key_fn(rec))
            except Exception:
                pass
    return done


def pooled_map(worker_fn, items, n_workers=2,
               stall_timeout_s=2400):
    '''Yield worker_fn(item) for all items, 2-process pool with a stall
watchdog: if no future completes within stall_timeout_s, workers are killed
and the remainder runs sequentially. Worker_fn must be picklable (an
importable function or a functools.partial thereof).'''
    from concurrent.futures import ProcessPoolExecutor, FIRST_COMPLETED, wait

    items = list(items)
    if len(items) <= 1 or n_workers <= 1:
        for it in items:
            yield worker_fn(it)
        return
    ctx = mp.get_context('spawn')
    ex = ProcessPoolExecutor(max_workers=n_workers, mp_context=ctx)
    done_count = 0
    try:
        futs = {ex.submit(worker_fn, it): it for it in items}
        pending = set(futs)
        while pending:
            done_set, pending = wait(pending, timeout=stall_timeout_s,
                                     return_when=FIRST_COMPLETED)
            if not done_set:
                raise RuntimeError(
                    f'pool stalled {stall_timeout_s}s with '
                    f'{len(pending)} futures pending')
            for f in done_set:
                done_count += 1
                yield f.result()
        ex.shutdown(wait=False, cancel_futures=True)
    except Exception as e:
        print(f'(pool yielded {done_count}/{len(items)} results, then '
              f'{type(e).__name__}; finishing remainder sequentially)',
              flush=True)
        for proc in (getattr(ex, '_processes', None) or {}).values():
            try:
                proc.kill()
            except Exception:
                pass
        ex.shutdown(wait=False, cancel_futures=True)
        for it in items[done_count:]:
            yield worker_fn(it)


In [ ]:

from sheafpatternfusion.workers import run_battery_row
from sheafpatternfusion.battery import aggregate_results

scored_path = OUT_DIR / 'null_battery_scored.jsonl'
done = load_done(scored_path, lambda r: r['instance_id'] + '|' + json.dumps(r['target']))
pending = [r for r in UNDECIDED
           if r['instance_id'] + '|' + json.dumps(r['target']) not in done]
print(f'{len(done)} scored on file, {len(pending)} to go')

if pending:
    tp0 = time.time()
    run_battery_row(dict(pending[0]))
    per = time.time() - tp0
    print(f'self-pilot: {per:.2f}s/row -> projected ~{per * len(pending) / 2 / 60:.1f} min '
          f'on 2 workers ({len(pending)} rows); continuing', flush=True)

fout = open(scored_path, 'a')
t0 = time.time()
completed = 0
for rec in pooled_map(run_battery_row, pending, n_workers=2):
    fout.write(json.dumps(rec) + '\n')
    fout.flush()
    completed += 1
    if completed % 200 == 0:
        el = time.time() - t0
        eta = el / completed * (len(pending) - completed)
        print(f'[{completed}/{len(pending)}] {el:.0f}s elapsed, ETA {eta:.0f}s', flush=True)
fout.close()

scored = [json.loads(l) for l in open(scored_path)]
out = aggregate_results(scored, BATTERY_CFG)
(OUT_DIR / 'null_battery.json').write_text(json.dumps(out['metrics'], indent=1))
with open(OUT_DIR / 'priority_sample.jsonl', 'w') as f:
    for r in out['priority_sample']:
        f.write(json.dumps(r) + '\n')
m = out['metrics']
print('S* size:', m['S_star_size'])
print('N0 acc:', round(m['N0_constant_recoverable']['accuracy'], 4),
      '| best N1:', m['best_swept']['N1'], '| best N3:', m['best_swept']['N3'])
print('BATTERY DONE')


In [ ]:
import glob
output_files = sorted(glob.glob(str(OUT_DIR / '*.jsonl')) + glob.glob(str(OUT_DIR / '*.json')))
for output_file in output_files:
    try:
        from google.colab import files
        files.download(output_file)
        print('Downloaded:', output_file)
    except Exception as e:
        print('(Not on Colab / download skipped):', e)
